# 14 - DANE operational simulation: four resolver policies

**CPU fine. Run all; idempotent.** Reads the trust-score predictions, the
TLS observations and the DNS/TLSA records for the frozen family-disjoint test
part of the probe universe, and simulates what a resolver would do to each
name under four policies. No model is trained.

| Policy | Rule |
|---|---|
| **A. Baseline** | resolve everything; no ML, no DANE |
| **B. DANE only** | look up TLSA for every name; validate where one exists; resolve everything else |
| **C. ML only** | block if calibrated score > t_high; resolve otherwise; no DANE |
| **D. ML + DANE (proposed)** | pass zone -> resolve; defer and block zones -> DANE/TLSA decides |

For policy D the deterministic decision is: a name with a DNSSEC-signed TLSA
association that matches the presented certificate is trusted; a name with a
TLSA association that does not match is blocked; a name with **no** TLSA
association cannot be validated and is treated as untrusted (*strict*). A
*permissive* variant (D') resolves un-validatable names in the defer zone but
logs them, and blocks only in the block zone; it is reported to show the
trade-off, not as the recommendation.

Reported per policy: malicious blocked (recall), benign blocked (FPR), share
of names requiring a DNSSEC/TLSA lookup, share receiving a cryptographic
validation, and a wall-clock cost proxy from the measured lookup latencies.
Thresholds come from validation (notebook 09); nothing is tuned on test.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install pyarrow zstandard

In [ ]:
import pandas as pd, numpy as np, json
from pathlib import Path
from sklearn.metrics import roc_curve
from src.evaluate import predictions
from src.utils.io import read_shards
from src.models.calibrate import Calibrator

PRED_DIR = Path(P['artifacts']['predictions']); TAB = Path(P['results']['tables']); FIG = Path(P['results']['figures'])
FD = 'family_disjoint_v1'; SEEDS = (42,43,44)

dnsr = read_shards(f"{P['data']['collected']}/dns_records", 'dns').drop_duplicates('domain', keep='last')
tls  = read_shards(f"{P['data']['collected']}/tls_probe", 'tls_probe').drop_duplicates('domain', keep='last')
print('dns records', len(dnsr), '| certificates', len(tls))

## TLSA validation outcome per name

For the few names that publish a TLSA record, the association is checked
against the collected certificate according to the record's selector and
matching type (RFC 6698): selector 0 = full certificate, 1 = SubjectPublicKeyInfo;
matching type 0 = exact, 1 = SHA-256, 2 = SHA-512. Everything else is
`no_tlsa`. This is the deterministic core's verdict, computed once.

In [ ]:
def tlsa_verdict(r):
    if not bool(r.get('has_tlsa', False)): return 'no_tlsa'
    if not bool(r.get('dnssec_signed', False)): return 'tlsa_unsigned'     # TLSA without DNSSEC is not DANE
    assoc = str(r.get('tlsa_cert_association', '') or '').lower()
    sel, mt = int(r.get('tlsa_selector', -1)), int(r.get('tlsa_matching_type', -1))
    if pd.isna(r.get('sha256_fingerprint')): return 'tlsa_no_cert'
    if mt == 1:
        ref = r['spki_sha256'] if sel == 1 else r['sha256_fingerprint']
        return 'match' if str(ref).lower() == assoc else 'mismatch'
    return 'tlsa_unverifiable'     # other selector/matching combinations not collected

base = dnsr.merge(tls[['domain','sha256_fingerprint','spki_sha256']], on='domain', how='left')
base['verdict'] = base.apply(tlsa_verdict, axis=1)
print(base['verdict'].value_counts().to_dict())

## Simulate

In [ ]:
def zones_from_val(split, seed):
    va = predictions.load(f'fusion_c_fused_{split}_s{seed}_VAL', PRED_DIR)
    hc = pd.read_parquet(f"{P['data']['features']}/fused_v1.parquet", columns=['domain','has_certificate'])
    va = va.merge(hc, on='domain', how='left'); va['has_certificate'] = va['has_certificate'].astype(bool)
    # same hybrid calibration as notebook 09 (Platt; per-regime for cert-holders, global otherwise)
    g = Calibrator('platt').fit(va['raw_score'], va['true_label'])
    vc = va.has_certificate.values
    c = Calibrator('platt').fit(va.loc[vc,'raw_score'], va.loc[vc,'true_label'])
    pv = np.where(vc, c.transform(va['raw_score']), g.transform(va['raw_score']))
    fpr, tpr, thr = roc_curve(va['true_label'], pv)
    t_low = float(thr[np.argmax(tpr >= 0.95)])
    ok = np.where(fpr <= 0.001)[0]; t_high = float(thr[ok[np.argmax(tpr[ok])]])
    return min(t_low, t_high), max(t_low, t_high)

LOOKUP_MS = float(base['lookup_ms'].median())      # measured DNS/TLSA lookup cost
INFER_MS = 0.0                                       # filled from table_latency.csv if present
lat = TAB/'table_latency.csv'
if lat.exists():
    L = pd.read_csv(lat); r = L[L.stage.str.contains('inference')]
    INFER_MS = float(r['median_ms'].iloc[0]) if len(r) else 0.0
print(f'lookup median {LOOKUP_MS:.0f} ms | inference {INFER_MS:.3f} ms')

def simulate(te, t_low, t_high):
    y = te['true_label'].values; sc = te['calibrated_score'].values; v = te['verdict'].values
    validated_ok = (v == 'match'); mismatch = (v == 'mismatch'); has_tlsa = np.isin(v, ['match','mismatch','tlsa_unverifiable','tlsa_unsigned'])
    zone = np.where(sc < t_low, 'pass', np.where(sc > t_high, 'block', 'defer'))
    pol = {}
    # A. baseline
    pol['A. baseline (no ML, no DANE)'] = dict(block=np.zeros(len(y),bool), lookup=np.zeros(len(y),bool), validate=np.zeros(len(y),bool), infer=False)
    # B. DANE only: lookup for all; block only on mismatch
    pol['B. DANE only'] = dict(block=mismatch, lookup=np.ones(len(y),bool), validate=has_tlsa, infer=False)
    # C. ML only: block above t_high
    pol['C. ML only (block > t_high)'] = dict(block=(zone=='block'), lookup=np.zeros(len(y),bool), validate=np.zeros(len(y),bool), infer=True)
    # D. ML + DANE strict: pass -> resolve; defer/block -> must validate; no association -> untrusted
    need = zone != 'pass'
    blockD = need & ~validated_ok
    pol['D. ML + DANE (strict)'] = dict(block=blockD, lookup=need, validate=need & has_tlsa, infer=True)
    # D'. permissive: defer resolves unless mismatch; block zone as strict
    blockDp = ((zone=='block') & ~validated_ok) | ((zone=='defer') & mismatch)
    pol["D'. ML + DANE (permissive defer)"] = dict(block=blockDp, lookup=need, validate=need & has_tlsa, infer=True)
    rows = []
    for name, p in pol.items():
        b = p['block']
        rows.append({'policy': name,
            'malicious_blocked (recall)': float(b[y==1].mean()),
            'benign_blocked (FPR)': float(b[y==0].mean()),
            'names_needing_TLSA_lookup': float(p['lookup'].mean()),
            'names_cryptographically_validated': float(p['validate'].mean()),
            'cost_ms_per_name': float(p['lookup'].mean()*LOOKUP_MS + (INFER_MS if p['infer'] else 0.0))})
    return pd.DataFrame(rows), zone

all_rows = []
for sd in SEEDS:
    te = predictions.load(f'trustscore_{FD}_s{sd}', PRED_DIR).merge(base[['domain','verdict']], on='domain', how='left')
    te['verdict'] = te['verdict'].fillna('no_tlsa')
    t_low, t_high = zones_from_val(FD, sd)
    tab, zone = simulate(te, t_low, t_high); tab['seed'] = sd; all_rows.append(tab)
res = pd.concat(all_rows)
summary = res.groupby('policy').agg(['mean','std']).round(4).drop(columns='seed', level=0)
display(summary)
summary.to_csv(TAB/'table_dane_policies.csv')
print('test names:', len(te), '| TLSA verdicts in test:', te['verdict'].value_counts().to_dict())

## The two numbers the paper needs

In [ ]:
m = res.groupby('policy').mean(numeric_only=True)
dane_only, mlonly, prop = m.loc['B. DANE only'], m.loc['C. ML only (block > t_high)'], m.loc['D. ML + DANE (strict)']
print(f"DANE-only blocks {100*dane_only['malicious_blocked (recall)']:.2f}% of malicious names while performing a TLSA lookup on 100% of names.")
print(f"ML+DANE (strict) blocks {100*prop['malicious_blocked (recall)']:.1f}% of malicious names (benign blocked {100*prop['benign_blocked (FPR)']:.2f}%) "
      f"while requiring a TLSA lookup on {100*prop['names_needing_TLSA_lookup']:.1f}% of names - "
      f"a {100*(1-prop['names_needing_TLSA_lookup']/dane_only['names_needing_TLSA_lookup']):.1f}% reduction in cryptographic lookups.")
print(f"ML only (hard block) reaches {100*mlonly['malicious_blocked (recall)']:.1f}% recall at {100*mlonly['benign_blocked (FPR)']:.2f}% benign blocked with no cryptographic check at all.")

## Figure: security benefit versus operational cost

In [ ]:
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'font.size':9,'font.family':'serif','axes.spines.top':False,'axes.spines.right':False,'savefig.dpi':300,'savefig.bbox':'tight'}) if 'matplotlib' in dir() else None
import matplotlib
matplotlib.rcParams.update({'font.size':9,'font.family':'serif','axes.spines.top':False,'axes.spines.right':False,'savefig.dpi':300,'savefig.bbox':'tight'})
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8))
order = list(m.index); x = np.arange(len(order))
axes[0].bar(x-0.2, m['malicious_blocked (recall)']*100, 0.4, label='malicious blocked', color='#B3432B')
axes[0].bar(x+0.2, m['benign_blocked (FPR)']*100, 0.4, label='benign blocked', color='#3B6B8F')
axes[0].set_xticks(x, [o.split('. ')[0] for o in order]); axes[0].set_ylabel('% of names'); axes[0].legend(frameon=False, fontsize=7)
axes[0].set_title('Security outcome', fontsize=9)
axes[1].bar(x-0.2, m['names_needing_TLSA_lookup']*100, 0.4, label='TLSA lookup required', color='#9AA5B1')
axes[1].bar(x+0.2, m['names_cryptographically_validated']*100, 0.4, label='validated (TLSA present)', color='#6B7B4C')
axes[1].set_xticks(x, [o.split('. ')[0] for o in order]); axes[1].set_ylabel('% of names'); axes[1].legend(frameon=False, fontsize=7)
axes[1].set_title('Cryptographic workload', fontsize=9)
fig.text(0.5, -0.06, '   '.join(f"{o.split('. ')[0]} = {o.split('. ',1)[1]}" for o in order), ha='center', fontsize=6.5)
fig.savefig(FIG/'fig9_dane_policies.pdf'); fig.savefig(FIG/'fig9_dane_policies.png'); plt.close(fig)
print('wrote fig9_dane_policies')

---
`table_dane_policies.csv` and `fig9_dane_policies` go into a new results
subsection (6.6, "Operational effect on DANE validation"). The TLSA coverage
table from `02c` (`table_dns_tlsa_coverage.csv`) goes into Section 4.3.